In [ ]:
!git clone -b step1-hybrid-retrieval https://github.com/WldEJackal/CodeFuse-CGM.git
%cd /teamspace/studios/travail/CodeFuse-CGM


In [ ]:
!pip install transformers==4.46.1 tokenizers==0.20.0 accelerate==1.0.1 peft==0.13.2 jinja2==3.1.2 fuzzywuzzy==0.18.0 python-Levenshtein==0.25.1 networkx==3.0

In [ ]:
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Cell 1: set base paths (Lightning AI)
BASE = "/teamspace/studios/this_studio/travail"
REPO_DIR = f"{BASE}/CodeFuse-CGM"
MODELS_DIR = f"{BASE}/models"
PKL_DIR = f"{REPO_DIR}/preprocess_embedding/tmp_node_embedding"

print("BASE:", BASE)
print("REPO_DIR:", REPO_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("PKL_DIR:", PKL_DIR)

In [ ]:
# Cell 4: download CGE-small into /teamspace/.../travail/models/cge-small
import os
from huggingface_hub import snapshot_download

os.makedirs(MODELS_DIR, exist_ok=True)

snapshot_download(
    repo_id="codefuse-ai/CodeFuse-CGE-Small",
    local_dir=f"{MODELS_DIR}/cge-small",
    local_dir_use_symlinks=False,
)
print("CGE-small downloaded to:", f"{MODELS_DIR}/cge-small")

In [ ]:
# Cell 5a (only if Qwen is gated/private): HF login prompt
from huggingface_hub import login
login()

In [ ]:
# Cell 6: download your PKL dataset from HF into preprocess_embedding/tmp_node_embedding
import os
from huggingface_hub import snapshot_download

os.makedirs(PKL_DIR, exist_ok=True)

snapshot_download(
    repo_id="Skyvokda/pkl_files",
    repo_type="dataset",
    local_dir=PKL_DIR,
    local_dir_use_symlinks=False,
)
print("PKLs downloaded to:", PKL_DIR)

In [ ]:
# Cell: download CodeGraph swe-bench-lite subset into /teamspace/.../travail/data/graphs/swe-bench-lite

import os
from huggingface_hub import snapshot_download

OUT_DIR = "/teamspace/studios/this_studio/travail/data"
os.makedirs(OUT_DIR, exist_ok=True)

snapshot_download(
    repo_id="codefuse-ai/CodeGraph",
    repo_type="dataset",
    local_dir=OUT_DIR,
    local_dir_use_symlinks=False, # download only the subset folder
)

print("Done. Listing:")
print(os.listdir(OUT_DIR)[:20])

In [ ]:
import os, glob

GRAPHS_DIR = "/teamspace/studios/this_studio/CodeFuse-Agent/data/graphs/swe-bench-lite"

print("GRAPHS_DIR exists:", os.path.exists(GRAPHS_DIR))
print("GRAPHS_DIR:", GRAPHS_DIR)

# Count json files directly inside the folder (flat layout)
flat_json = sorted(glob.glob(os.path.join(GRAPHS_DIR, "*.json")))
print("\nFlat JSON files in GRAPHS_DIR:", len(flat_json))
print("Sample flat files:", [os.path.basename(x) for x in flat_json[:5]])

# Check if there are subfolders (nested layout)
subdirs = sorted([d for d in glob.glob(os.path.join(GRAPHS_DIR, "*")) if os.path.isdir(d)])
print("\nSubdirectories in GRAPHS_DIR:", len(subdirs))
print("Sample subdirs:", [os.path.basename(x) for x in subdirs[:5]])

# If nested, count json files inside subfolders (one level deep)
nested_json = sorted(glob.glob(os.path.join(GRAPHS_DIR, "*", "*.json")))
print("\nNested JSON files (one level deep):", len(nested_json))
print("Sample nested files:", ["/".join(x.split("/")[-2:]) for x in nested_json[:5]])

# Decide layout
if len(flat_json) > 0 and len(subdirs) == 0:
    print("\nDETECTED LAYOUT: A) flat folder of repo graph json files")
elif len(nested_json) > 0:
    print("\nDETECTED LAYOUT: B) nested folder layout (repo subfolders)")
else:
    print("\nDETECTED LAYOUT: unknown/empty")

In [ ]:
import os

BASE = "/teamspace/studios/this_studio/CodeFuse-CGM"
os.environ["PYTHONPATH"] = ":".join([
    BASE,
    f"{BASE}/retriever",
    f"{BASE}/reranker",
    os.environ.get("PYTHONPATH", "")
]).strip(":")

print("PYTHONPATH =", os.environ["PYTHONPATH"])

In [ ]:
BASE = "/teamspace/studios/this_studio/travail/CodeFuse-CGM"
PROMPTS = f"{BASE}/test_rewriter_prompt.json"
RAW_OUT = f"{BASE}/test_rewriter_raw.json"
FINAL_OUT = f"{BASE}/rewriter_output.json"
print(PROMPTS, RAW_OUT, FINAL_OUT)

In [ ]:
# Notebook cell: download Qwen 7B into /teamspace/studios/this_studio/travail/models/qwen-7b

import os
from huggingface_hub import snapshot_download, login

# If Qwen is gated/private, run login() once:
# login()

LOCAL_DIR = "/teamspace/studios/this_studio/travail/models/qwen-7b"
os.makedirs(LOCAL_DIR, exist_ok=True)

snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir=LOCAL_DIR,
    local_dir_use_symlinks=False,
)

print("Done. Sample files:", os.listdir(LOCAL_DIR)[:10])

In [ ]:
!pip install pickleshare
!pip install pand

In [ ]:
%cd /teamspace/studios/this_studio/travail/CodeFuse-CGM
!python rewriter/generate_rewriter_prompt.py
!ls -la /teamspace/studios/this_studio/travail/CodeFuse-CGM/rewriter/test_rewriter_prompt.json

In [ ]:
%cd /teamspace/studios/this_studio/travail/CodeFuse-CGM
!python rewriter/inference_rewriter.py --prompt_path rewriter/test_rewriter_prompt.json --batch_size 16

In [ ]:
%cd /teamspace/studios/this_studio/CodeFuse-CGM
!python preprocess_embedding/generate_code_content.py

In [ ]:
!python preprocess_embedding/generate_code_embedding.py

In [ ]:
!pip install deepspeed


In [ ]:
import json

path = "/teamspace/studios/this_studio/travail/CodeFuse-CGM/rewriter/test_rewriter_output.json"
with open(path, "r", encoding="utf-8") as f:
    d = json.load(f)

first_id = next(iter(d.keys()))
print("First instance_id:", first_id)
print("Keys:", d[first_id].keys())
print("Value sample:", d[first_id])

In [ ]:
!pip install sentencepiece

In [ ]:
%cd /teamspace/studios/this_studio/travail/CodeFuse-CGM
!python rewriter/generate_rewriter_prompt.py

In [ ]:
!python /teamspace/studios/this_studio/travail/CodeFuse-CGM/preprocess_embedding/generate_rewriter_embedding.py

In [ ]:
!python -m pip install -U faiss-cpu

In [ ]:
import re, pickle, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

BASE = "/teamspace/studios/this_studio/travail/CodeFuse-CGM"
IN_JSON = f"{BASE}/rewriter/test_rewriter_output.json"
OUT_PKL = f"{BASE}/rewriter/rewriter_embedding.pkl"   # overwrite
MODEL_PATH = "/teamspace/studios/this_studio/travail/models/cge-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_QUERIES = 5
MAX_LEN = 256
BATCH = 8

def extract_queries(text: str, max_q=5):
    if not isinstance(text, str):
        return []
    m = re.search(r"\[start_of_related_queries\](.*?)\[end_of_related_queries\]", text, re.DOTALL | re.IGNORECASE)
    block = m.group(1).strip() if m else text.strip()
    lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
    qs = []
    for ln in lines:
        ln = re.sub(r"^(query\s*\d+\s*:\s*)", "", ln, flags=re.IGNORECASE)
        ln = re.sub(r"^[-*\d\.\)]\s*", "", ln)
        if len(ln) >= 8:
            qs.append(ln)
    # dedup
    out, seen = [], set()
    for q in qs:
        if q not in seen:
            out.append(q); seen.add(q)
    return out[:max_q]

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

df = pd.read_json(IN_JSON)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True, use_fast=False)
cge = AutoModel.from_pretrained(MODEL_PATH, trust_remote_code=True, torch_dtype=torch.float16).to(DEVICE)
cge.eval()
plm = cge.plm_model
plm.eval()

out = {}

for _, row in tqdm(df.iterrows(), total=len(df), desc="Embed queries to 3072 (plm)"):
    iid = str(row["instance_id"])
    infer = row.get("rewriter_inferer", "")
    queries = extract_queries(infer, MAX_QUERIES)
    if not queries and isinstance(infer, str) and infer.strip():
        queries = [infer.strip()[:2000]]
    if not queries:
        continue

    # batch embed queries for this instance
    embs = []
    with torch.no_grad():
        for i in range(0, len(queries), BATCH):
            batch_q = queries[i:i+BATCH]
            toks = tokenizer(batch_q, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
            toks = {k: v.to(DEVICE) for k, v in toks.items()}
            outputs = plm(input_ids=toks["input_ids"], attention_mask=toks["attention_mask"],
                          output_hidden_states=True, return_dict=True)
            last_hidden = outputs.hidden_states[-1]          # [B,T,H] where H should be 3072
            emb = mean_pool(last_hidden, toks["attention_mask"])  # [B,H]
            embs.append(emb.detach().cpu().numpy())

    emb = np.concatenate(embs, axis=0).astype(np.float32)
    out[iid] = emb

with open(OUT_PKL, "wb") as f:
    pickle.dump(out, f)

# verify dims
k0 = next(iter(out.keys()))
print("Saved:", OUT_PKL)
print("Sample:", k0, out[k0].shape)

In [ ]:
import pickle, numpy as np, os

BASE = "/teamspace/studios/this_studio/travail/CodeFuse-CGM"

QUERY_PKL = f"{BASE}/rewriter/rewriter_embedding.pkl"
NODE_DIR = f"{BASE}/preprocess_embedding/tmp_node_embedding/tmp_node_embedding"

print("QUERY_PKL exists:", os.path.exists(QUERY_PKL), QUERY_PKL)
q = pickle.load(open(QUERY_PKL, "rb"))
kq = next(iter(q.keys()))
print("Query sample key:", kq)
print("Query shape:", np.array(q[kq]).shape)

pkls = [x for x in os.listdir(NODE_DIR) if x.endswith(".pkl")]
print("Node PKL count:", len(pkls))
p0 = os.path.join(NODE_DIR, pkls[0])
node = pickle.load(open(p0, "rb"))
nv = next(iter(node["code"].values()))
print("Node sample pkl:", pkls[0])
print("Node vec dim:", np.array(nv).shape)

In [ ]:

!python /teamspace/studios/this_studio/travail/CodeFuse-CGM/retriever/locate_anchor_node.py

In [ ]:
!python /teamspace/studios/this_studio/travail/CodeFuse-CGM/retriever/subgraph.py

In [ ]:
!python /teamspace/studios/this_studio/travail/CodeFuse-CGM/retriever/serialize_subgraph.py

In [ ]:
%cd /teamspace/studios/this_studio/travail/CodeFuse-CGM
!python reranker/reranker.py --stage_1_k 10 --stage_2_k 5

In [10]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 221.9 MB/s  0:00:00m0:00:01
  Attempting uninstall: fsspec90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [pyarrow]
    Found existing installation: fsspec 2026.1.0━━━━━━━━━━━━━━ 1/6 [pyarrow]
    Uninstalling fsspec-2026.1.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [pyarrow]
      Successfully uninstalled fsspec-2026.1.0━━━━━━━━━━━━━━━━ 1/6 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [datasets]5/6 [datasets]ess]


In [11]:
%cd /teamspace/studios/this_studio/travail/CodeFuse-CGM
!python reranker/generate_groundtruth.py


/teamspace/studios/this_studio/travail/CodeFuse-CGM


🚀 Téléchargement de SWE-bench_Lite depuis Hugging Face...
README.md: 3.67kB [00:00, 23.0MB/s]
data/dev-00000-of-00001.parquet: 100%|████████| 120k/120k [00:00<00:00, 387kB/s]
data/test-00000-of-00001.parquet: 100%|████| 1.12M/1.12M [00:00<00:00, 4.46MB/s]
Generating test split: 100%|████████| 300/300 [00:00<00:00, 28271.12 examples/s]
✅ Dataset téléchargé : 300 instances.

🎉 SUCCÈS TOTAL !
📂 Fichier Ground Truth créé ici : C:\Users\clouduser\Desktop\CodeFuse-cgm\ground_truth.json
Exemple (astropy__astropy-12907) : ['astropy/modeling/separable.py']


In [13]:
%cd /teamspace/studios/this_studio/travail/CodeFuse-CGM
!python reranker/eval.py


/teamspace/studios/this_studio/travail/CodeFuse-CGM


Loading predictions...
Prediction json files found: 43
Sample files: ['django__django-11620.json', 'django__django-12700.json', 'django__django-11133.json', 'django__django-12589.json', 'django__django-11848.json']
Loaded predictions: 43
Sample pred instance: django__django-11620
Sample pred files: ['django/urls/base.py', 'django/core/exceptions.py']

Loading ground truth...
Loaded ground truth: 300

       EVALUATION RESULTS     
Evaluated instances (intersection): 43
Recall@1    : 41.86%
Recall@5    : 44.19%
------------------------------
Precision@1 : 41.86%
Precision@5 : 8.84%
------------------------------
MRR@5       : 0.4302
